## ⚙️ Setup Instructions

Before running this notebook:

### 1. Install PyYAML (if not already installed):
```bash
pip install pyyaml
```

### 2. Configure paths in `config/config.yaml`:

- Edit `data.enhanced_features` to point to your data
- Edit other paths as needed

### 3. Ensure directory structure:
```
NMDpredictionmodel/
├── Model/
│   └── CatBoost/
│       ├── config/
│       │   └── config.yaml          ← Edit this file
│       └── notebooks/
│           └── 01_data_loading_and_merging.ipynb    ← You are here
```

### 4. Run cells in order!

---

## 📁 Expected File Locations

This notebook expects the following structure (edit in `config/config.yaml`):

**Input files** (you need these):
- Enhanced features CSV
- Codon optimality TSV  
- Half-life features CSV
- Readthrough scores CSV

**Output file** (will be created):
- `data/processed/TOPMed_merged_with_codon_optimality.csv`

---

## 🚨 Troubleshooting

**Error: "FileNotFoundError: config.yaml"**
- Check that `config/config.yaml` exists
- Verify the path in the code matches your directory structure

**Error: "FileNotFoundError: [your data file]"**
- Edit paths in `config/config.yaml` to match your file locations

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import yaml

## Configuration

In [ ]:
# Load configuration
config_path = Path("../config/config.yaml")  # Assumes notebooks/ is one level down from config/
with open(config_path) as f:
    config = yaml.safe_load(f)

# Input paths from config
PATH_ENHANCED = config['data']['enhanced_features']
PATH_CODON_OPT = config['data']['codon_optimality']
PATH_HALFLIFE = config['data']['halflife']
PATH_READTHROUGH = config['data']['readthrough']

# Output path from config
PATH_OUTPUT = config['data']['merged']

print("✓ Configuration loaded from:", config_path)
print(f"\nInput files:")
print(f"  Enhanced:    {PATH_ENHANCED}")
print(f"  Codon opt:   {PATH_CODON_OPT}")
print(f"  Half-life:   {PATH_HALFLIFE}")
print(f"  Readthrough: {PATH_READTHROUGH}")
print(f"\nOutput:")
print(f"  {PATH_OUTPUT}")

## 1. Load All Datasets

In [ ]:
print("="*80)
print("LOADING DATASETS")
print("="*80)

# Load enhanced features (has everything except correct codon features)
df_enhanced = pd.read_csv(PATH_ENHANCED)
print(f"✓ Enhanced features: {df_enhanced.shape}")

# Load codon optimality data
df_codon = pd.read_csv(PATH_CODON_OPT, sep='\t')
print(f"✓ Codon optimality: {df_codon.shape}")

# Load half-life (PC1) data
df_halflife = pd.read_csv(PATH_HALFLIFE)
print(f"✓ Half-life (PC1): {df_halflife.shape}")

# Load readthrough data
df_readthrough = pd.read_csv(PATH_READTHROUGH)
print(f"✓ Readthrough: {df_readthrough.shape}")

## 2. Remove indel and Standardize Keys

In [ ]:
print("\n" + "="*80)
print("IDENTIFYING INDEL KEYS FROM READTHROUGH DATA")
print("="*80)

indel_keys = set()

if 'V7' in df_readthrough.columns and 'V8' in df_readthrough.columns:
    print(f"\nScanning readthrough data for indels...")
    
    # Identify rows with indels
    indel_mask = ((df_readthrough['V7'].astype(str).str.contains('-', na=False)) | 
                  (df_readthrough['V8'].astype(str).str.contains('-', na=False)))
    
    indels_v7 = df_readthrough['V7'].astype(str).str.contains('-', na=False).sum()
    indels_v8 = df_readthrough['V8'].astype(str).str.contains('-', na=False).sum()
    total_indels = indel_mask.sum()
    
    print(f"  Indels detected:")
    print(f"    V7 column: {indels_v7}")
    print(f"    V8 column: {indels_v8}")
    print(f"    Total rows with indels: {total_indels}")
    
    # Extract keys of indel variants
    indel_keys = set(df_readthrough.loc[indel_mask, 'key'].values)
    
    print(f"\n✓ Identified {len(indel_keys)} unique indel variant keys")
    print(f"  Sample indel keys: {list(indel_keys)[:3]}")
else:
    print("  ⚠️  V7/V8 columns not found - cannot identify indels")

print("\n" + "="*80)
print("FILTERING INDELS FROM ALL DATASETS")
print("="*80)

if len(indel_keys) > 0:
    # Filter enhanced dataset
    before = len(df_enhanced)
    df_enhanced = df_enhanced[~df_enhanced['key'].isin(indel_keys)]
    removed = before - len(df_enhanced)
    print(f"\nEnhanced dataset:")
    print(f"  Before: {before} variants")
    print(f"  After: {len(df_enhanced)} variants")
    print(f"  Removed: {removed} indels")
    
    # Filter codon optimality dataset based on PTC_ID format
    if 'PTC_ID' in df_codon.columns:
        before = len(df_codon)
        
        # Parse PTC_ID to check ref and alt allele lengths
        # Format: chr_pos_ref_alt
        def is_snv(ptc_id):
            parts = ptc_id.split('_')
            if len(parts) == 4:
                ref = parts[2]
                alt = parts[3]
                # SNV = both ref and alt are single nucleotides
                return len(ref) == 1 and len(alt) == 1
            return False
        
        snv_mask = df_codon['PTC_ID'].apply(is_snv)
        indels_in_codon = (~snv_mask).sum()
        
        print(f"\nCodon optimality - filtering by allele length:")
        print(f"  Total variants: {before}")
        print(f"  SNVs (len=1 for both ref/alt): {snv_mask.sum()}")
        print(f"  Indels (len>1 for ref or alt): {indels_in_codon}")
        
        if indels_in_codon > 0:
            print(f"  Sample indel PTC_IDs: {df_codon.loc[~snv_mask, 'PTC_ID'].head(3).tolist()}")
        
        df_codon = df_codon[snv_mask]
        removed = before - len(df_codon)
        
        print(f"  After filtering: {len(df_codon)} variants")
        print(f"  Removed: {removed} indels")
        
        # NOW create key column from cleaned SNV data only
        def convert_ptc_id_to_key(ptc_id):
            parts = ptc_id.split('_')
            chrom = parts[0]
            pos = parts[1]
            ref = parts[2]
            alt = parts[3]
            return f"{chrom}:{pos}_{ref}>{alt}"
        
        df_codon['key'] = df_codon['PTC_ID'].apply(convert_ptc_id_to_key)
        print(f"  ✓ Created 'key' from {len(df_codon)} SNVs")
        print(f"  Sample keys: {df_codon['key'].head(3).tolist()}")
    
    # Filter half-life dataset
    before = len(df_halflife)
    df_halflife = df_halflife[~df_halflife['key'].isin(indel_keys)]
    removed = before - len(df_halflife)
    print(f"\nHalf-life dataset:")
    print(f"  Before: {before} variants")
    print(f"  After: {len(df_halflife)} variants")
    print(f"  Removed: {removed} indels")
    
    # Filter readthrough dataset
    before = len(df_readthrough)
    df_readthrough = df_readthrough[~df_readthrough['key'].isin(indel_keys)]
    removed = before - len(df_readthrough)
    print(f"\nReadthrough dataset:")
    print(f"  Before: {before} variants")
    print(f"  After: {len(df_readthrough)} variants")
    print(f"  Removed: {removed} indels")
    
    print(f"\n✓ Successfully filtered indel variants from all datasets")
else:
    print("\n  No indels to filter")

## 3. Remove Old Features to be Replaced

In [ ]:
print("\n" + "="*80)
print("REMOVING OLD FEATURES")
print("="*80)

# Drop AverageCodonRNAUsage (wrong data)
if 'AverageCodonRNAUsage' in df_enhanced.columns:
    print(f"\n✓ Dropping AverageCodonRNAUsage (incorrect data)")
    print(f"  Old values (first 5): {df_enhanced['AverageCodonRNAUsage'].head().tolist()}")
    df_enhanced = df_enhanced.drop(columns=['AverageCodonRNAUsage'])
else:
    print(f"\n⚠️  AverageCodonRNAUsage not found (already dropped?)")

# Drop median_half_life if present (replaced by half_life_PC1)
if 'median_half_life' in df_enhanced.columns:
    print(f"✓ Dropping median_half_life (replaced by PC1)")
    df_enhanced = df_enhanced.drop(columns=['median_half_life'])
else:
    print(f"⚠️  median_half_life not found (already dropped?)")

print(f"\nEnhanced dataset after drops: {df_enhanced.shape}")

## 4. Merge Codon Optimality Features

In [ ]:
print("\n" + "="*80)
print("MERGING CODON OPTIMALITY FEATURES")
print("="*80)

# Select columns to merge
codon_cols = ['key', 'CodonOptimalityFraction_CDS', 'CodonOptimalityFraction_PTCpm100nt']
print(f"\nMerging columns: {codon_cols}")

# Check which variants will match
common_keys = set(df_enhanced['key']) & set(df_codon['key'])
print(f"\nKey overlap: {len(common_keys)}/{len(df_enhanced)} variants")

# Merge
df_result = df_enhanced.merge(
    df_codon[codon_cols],
    on='key',
    how='left'
)

print(f"\n✓ Merged codon optimality features")
print(f"  Shape after merge: {df_result.shape}")

# Check merge results
for col in ['CodonOptimalityFraction_CDS', 'CodonOptimalityFraction_PTCpm100nt']:
    matched = df_result[col].notna().sum()
    pct = matched/len(df_result)*100
    print(f"  {col}: {matched}/{len(df_result)} ({pct:.1f}%)")
    if matched > 0:
        print(f"    Mean: {df_result[col].mean():.4f}")
        print(f"    Range: {df_result[col].min():.4f} - {df_result[col].max():.4f}")

## 5. Merge Half-Life (PC1) Features

In [ ]:
print("\n" + "="*80)
print("MERGING HALF-LIFE (PC1) FEATURES")
print("="*80)

if 'half_life_PC1' in df_halflife.columns:
    # Merge half-life PC1
    df_result = df_result.merge(
        df_halflife[['key', 'half_life_PC1']],
        on='key',
        how='left'
    )
    
    matched = df_result['half_life_PC1'].notna().sum()
    print(f"\n✓ Merged half_life_PC1: {matched}/{len(df_result)} matched")
    print(f"  Mean: {df_result['half_life_PC1'].mean():.4f}")
    print(f"  Sample values: {df_result['half_life_PC1'].head().tolist()}")
else:
    print(f"\n⚠️  half_life_PC1 not found in dataset")

print(f"\nShape after merge: {df_result.shape}")

## 6. Merge Readthrough Features

In [ ]:
print("\n" + "="*80)
print("MERGING READTHROUGH FEATURES")
print("="*80)

# Check available readthrough columns
readthrough_cols = [col for col in df_readthrough.columns if 'readthrough' in col.lower()]
print(f"\nAvailable readthrough columns: {readthrough_cols}")

# Merge readthrough features
cols_to_merge = ['key', 'readthrough_score_hek293t', 'readthrough_category_hek293t']
cols_to_merge = [c for c in cols_to_merge if c in df_readthrough.columns]

print(f"Merging columns: {cols_to_merge}")

df_result = df_result.merge(
    df_readthrough[cols_to_merge],
    on='key',
    how='left'
)

print(f"\n✓ Merged readthrough features")
print(f"  Shape after merge: {df_result.shape}")

# Check merge results
for col in cols_to_merge:
    if col != 'key':
        matched = df_result[col].notna().sum()
        pct = matched/len(df_result)*100
        print(f"  {col}: {matched}/{len(df_result)} ({pct:.1f}%)")
        
        if 'category' in col:
            print(f"    Distribution:")
            print(df_result[col].value_counts().to_string(header=False))
        elif matched > 0:
            print(f"    Mean: {df_result[col].mean():.2f}")
            print(f"    Range: {df_result[col].min():.2f} - {df_result[col].max():.2f}")

## 7. Final Verification

In [ ]:
print("\n" + "="*80)
print("FINAL VERIFICATION")
print("="*80)

print(f"\nFinal dataset shape: {df_result.shape}")
print(f"  Original enhanced: {df_enhanced.shape[1]} columns")
print(f"  Final: {df_result.shape[1]} columns")
print(f"  Net change: {df_result.shape[1] - df_enhanced.shape[1]:+d} columns")

# Verify old features are gone
print("\n✓ Removed features:")
if 'AverageCodonRNAUsage' not in df_result.columns:
    print("  - AverageCodonRNAUsage (dropped)")
if 'median_half_life' not in df_result.columns:
    print("  - median_half_life (dropped)")

# Verify new features are present
print("\n✓ New features added:")
for col in ['CodonOptimalityFraction_CDS', 'CodonOptimalityFraction_PTCpm100nt', 
            'half_life_PC1', 'readthrough_score_hek293t', 'readthrough_category_hek293t']:
    if col in df_result.columns:
        non_null = df_result[col].notna().sum()
        print(f"  - {col}: {non_null}/{len(df_result)} non-null")

# Check for any NaN issues
print(f"\nMissing data summary:")
missing_summary = df_result.isnull().sum()
missing_summary = missing_summary[missing_summary > 0].sort_values(ascending=False)
if len(missing_summary) > 0:
    print(f"  Columns with missing data: {len(missing_summary)}")
    print(f"  Top 10 columns with most missing:")
    for col, count in missing_summary.head(10).items():
        pct = count/len(df_result)*100
        print(f"    {col}: {count} ({pct:.1f}%)")
else:
    print("  No missing data!")

## 8. Save Merged Dataset

In [ ]:
print("\n" + "="*80)
print("SAVING MERGED DATASET")
print("="*80)

df_result.to_csv(PATH_OUTPUT, index=False)
print(f"\n✓ Saved: {PATH_OUTPUT}")
print(f"  Rows: {len(df_result)}")
print(f"  Columns: {df_result.shape[1]}")

print("\n" + "="*80)
print("COMPLETE! 🎉")
print("="*80)
print("\nReady for feature cleaning and selection (next notebook)")